In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
Data = pd.read_csv('/kaggle/input/q1-ka-ai-2026/Q1_data.csv')
# read the file Q1_data.csv from the directory in the path

In [ ]:
Data.head()

In [ ]:
Data.info()

In [ ]:
Data.describe()

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.hist(Data['Delivery_Time'], bins=60, edgecolor='black')
plt.title('Distribution of Delivery Time')
plt.xlabel('Dellivr time')
plt.ylabel('Count')
plt.show()

# Show the Distribution of the Target Label (i think it is a normal distribution so no need to transform the target)
# but thier some outleir

In [ ]:
Data.drop(columns='Order_ID',inplace=True)
Data.head(1)


In [ ]:
def check_missing_values(df):
    missing_values = df.isnull().sum()
    print("Missing Values per Column:")
    print(missing_values[missing_values > 0])
    if missing_values.any():
        print("\nHandle Missing Values as needed.")
    else:
        print("\nNo Missing Values Found.")

check_missing_values(Data) # After checke missing value we can handling them by :
# First : Drop rows where target (Delivry Time) - can't predict without them
print(f"Before: {Data.shape}")
Data = Data.dropna(subset=['Delivery_Time'])
print(f"After dropping missing Delivery_Time: {Data.shape}")

# Second : filling the missing value of weather,Traffic_level, Time_of_day ..elt
for col in ['Time_of_Day', 'Traffic_Level', 'Weather']:
    Data[col] = Data[col].fillna('unknown')
Data['Courier_Experience_yrs'] = Data['Courier_Experience_yrs'].fillna(Data['Courier_Experience_yrs'].mode()[0])

In [ ]:
# Handel duplecated sampel
def check_duplicates(df):
    duplicates = df.duplicated().sum()
    print(f"Number of Duplicate Samples: {duplicates}")
    if duplicates > 0:
        print("Dropping Duplicates...")
        df.drop_duplicates(inplace=True)
        print("Duplicates Dropped.")
    else:
        print("No Duplicate Samples Found.")
check_duplicates(Data)

In [ ]:
# Encodeng
from sklearn.preprocessing import StandardScaler, OneHotEncoder,LabelEncoder
categorical_cols = ['Weather'	,'Traffic_Level','Time_of_Day','Vehicle_Type']
for col in categorical_cols:
    le = LabelEncoder()
    Data[col] = le.fit_transform(Data[col].astype(str))

Data.head()

In [ ]:
# prepearing data for scaling and model.
from sklearn.model_selection import train_test_split, KFold
feature_cols = ['Distance_km' , 'Weather' ,'Traffic_Level', 'Time_of_Day', 'Vehicle_Type'
                ,'Preparation_Time_min' ,'Courier_Experience_yrs']
X = Data[feature_cols]
y = Data['Delivery_Time']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# scaling the data :
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nScaled ranges - Min: {X_train_scaled.min():.2f}, Max: {X_train_scaled.max():.2f}")
pd.DataFrame(X_train_scaled, columns=X_train.columns).head(3)

In [ ]:
# we don't need any type of Check for target imbalance because it is not Classification problem
# if it is we can do the suffeling to have better result.

In [ ]:
# splited already befor scaling

In [ ]:
from sklearn.model_selection import StratifiedKFold

# Show full dataset class distribution
full_ratio = (y.value_counts(normalize=True) * 100).sort_index()
print("Full Dataset Class Distribution")
print("  y class percentages:", {k: f"{v:.2f}%" for k, v in full_ratio.items()})
print("-" * 40)

# Define Stratified K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Stratified K-Fold Cross Validation\n" + "-"*40)

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    print(f"Fold {fold}")
    print("  X_train shape:", X_train.shape)
    print("  X_test shape :", X_test.shape)
    print("  y_train shape:", y_train.shape)
    print("  y_test shape :", y_test.shape)

    # showing class distribution
    train_ratio = (y_train.value_counts(normalize=True) * 100).sort_index()
    test_ratio = (y_test.value_counts(normalize=True) * 100).sort_index()

    print("  y_train class percentages:", {k: f"{v:.2f}%" for k, v in train_ratio.items()})
    print("  y_test class percentages :", {k: f"{v:.2f}%" for k, v in test_ratio.items()})

    print("-" * 40)


In [ ]:
# Train Random Forest Regressor
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train)
print("Model trained!")

In [ ]:
# Predict and evaluate
from sklearn.metrics import mean_absolute_error
y_pred = model.predict(X_test_scaled)
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE: {mae:,.2f}")

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Plot for Linear Regression Predictions vs. Ground Truth
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_pred)], 'r--', linewidth=2)

plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here: